In [59]:
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', None)

In [60]:
lines = []
with open('nouns.txt', 'r') as nouns:
    for x in nouns:
        new_line = x.split('\t')
        if len(new_line) > 1:
            new_line[2] = new_line[2][:-1]
            lines.append(new_line)
for x in lines:
    if '#' in x[2]:
        print(x)

['104. Breath', 'Der Atem', '- ###Changed the plural (die Atemzüge) to -, as it expresses something quite different']
['358. Smoke', 'Der Rauch', 'Die Rauche ###Corrected the plural from "Rauch"']
['429. Future', 'Die Zukunft', 'Die Zukünfte ###Corrected plural from die Zukunft']
['481. Television', 'Der Fernseher', 'Die Fernseher ### Nonsense; Der Fernseher is the starting point (rather than das Fernsehen - watching TV)']
['485. Lap', 'Der Schoss', 'Die Schöße ###Corrected the singular from der Schoß, and the plural from die Schoß']
['495. Ghost', 'Der Geist', 'Die Geister ###Corrected from Das']
['683. Return', 'Die Rückkehr', '- ###Corrected - no plural']
['785. Twin', 'Der Zwilling', 'Die Zwillinge ###Corrected from Das']
['983. Fur', 'Der Pelz', 'Die Pelze  ###Corrected plural from die Felle']
['1065. Lightning', 'Der Blitz', 'Die Blitze ### Corrected from Das']
['1223. Trade', 'Der Handel', 'Die Händel ###Corrected plural from Handel']
['1285. Shelter', 'Der Schutz', 'Die Schutze

In [61]:
###Germanization, part I: on the plurals for nouns
def normalize_data(data):
    normalized_data = []
    for x in data:
        new_plural = ' '.join(x[2].split(' ')[:2]) if '#' in x[2] else x[2]
        if new_plural.startswith('-'):
            new_plural = '-'
        if new_plural in('(usually uncountable)', '(usually unCountable)', '(no plural form)', '(plural noun)'):
            new_plural = '-'
        if len(x[1].split(' '))>2: #Should be one case
            continue
        new_line = [*x[0].split(' '), x[1], new_plural]
        normalized_data.append(new_line)
    return normalized_data
        
lines = normalize_data(lines)

In [62]:
#Preparations for radical pluralism: Purge '-'
def purge_dashes(data):
    dashless_data = []
    for x in data:
        if x[3] == '-' or x[3] == '- ':
            continue
        dashless_data.append(x)
    return dashless_data

dashless_lines = purge_dashes(lines)

In [63]:
#Preparations for radical pluralism: Purge duplicates
def purge_pure_duplicates(data):
    all_german_nouns = set()
    possible_plurals = []
    
    for x in data:
        if x[2] in all_german_nouns:
            continue
        all_german_nouns.add(x[2])
        possible_plurals.append(x)
    return possible_plurals

unduplicated_lines = purge_pure_duplicates(dashless_lines)

In [64]:
exceptions_for_complexes = """
    Der ort  ;  Der Sport
    Der arm  ;  Der Alarm
    Der rat  ;  Der Pirat
    Der ast  ;  Der Toast
    Der rand  ;  Der Strand
    Der ast  ;  Der Palast
    Der halt  ;  Der Inhalt
    Der puls  ;  Der Impuls
    Der bus  ;  Der Globus
    Der ton  ;  Der Karton
    Der rat  ;  Der Verrat
    Die asche  ;  Die Flasche
    Die ecke  ;  Die Strecke
    Die lanze  ;  Die Pflanze
    Die form  ;  Die Uniform
    Der akt  ;  Der Kontakt
    Der ort  ;  Der Komfort
    Das wesen  ;  Das Anwesen
    Der stand  ;  Der Bestand
    Der hof  ;  Der Bischof
    Die mode  ;  Die Kommode
    Der test  ;  Der Protest
    Der rumpf  ;  Der Strumpf
    Der arm  ;  Der Schwarm
    Der text  ;  Der Kontext
    Die aktion  ;  Die Reaktion
    Der stall  ;  Der Kristall
    Das gen  ;  Das Versagen
    Der halter  ;  Der Schalter
    Der graf  ;  Der Fotograf
    Die vision  ;  Die Division
    Der ast  ;  Der Kontrast
    Der sprung  ;  Der Ursprung
    Die welle  ;  Die Schwelle
    Der wurf  ;  Der Maulwurf
    Die rolle  ;  Die Kontrolle
    Das gen  ;  Das Vergnügen
    Die form  ;  Die Plattform
    Der ort  ;  Der Transport
    Der sport  ;  Der Transport
    Der burger  ;  Der Hamburger
    Die weite  ;  Die Reichweite
    Der stand  ;  Der Widerstand
    Der schlag  ;  Der Herzschlag
    Das skript  ;  Das Manuskript
    Die aktion  ;  Die Attraktion
    Die mission  ;  Die Kommission
    Der teller  ;  Der Hersteller
    Der rauch  ;  Der Missbrauch
    Die formation  ;  Die Information
    Der teig  ;  Der Bürgersteig
    Die nation  ;  Die Kombination
    Das tier  ;  Das Hauptquartier
"""

unpurgables = set()
for exc in exceptions_for_complexes.split('\n'):
    if ';' in exc:
        unpurgables.add(exc.split(';')[1][2:])

In [65]:
#Manual operation may be necessary
def observe_double_words(data):
    all_german_nouns = set()
    all_german_nouns_listed = []
    data = sorted(data, key=lambda x: len(x[2]))
    for x in data:
        dead = False
        gender, genderless = x[2].split(' ')
        if isinstance(genderless, tuple):
            genderless = genderless[-1]
        for suffix in range(3, len(genderless)-1):
            if gender + ' ' + genderless[-suffix:] in all_german_nouns and gender + ' ' + genderless not in unpurgables:
                dead = True
        if not dead:
            all_german_nouns_listed.append(x)
            all_german_nouns.add(gender + ' ' + genderless[0].lower() + genderless[1:])
    print(all_german_nouns)
    return all_german_nouns_listed
purged_plurality = observe_double_words(unduplicated_lines)

{'Die erwähnung', 'Der sumpf', 'Der krieg', 'Der kloß', 'Das hotel', 'Der therapeut', 'Der gegner', 'Das korn', 'Die region', 'Der schlag', 'Der satz', 'Die gemeinschaft', 'Die brille', 'Die gnade', 'Die mama', 'Der verbündete', 'Der clan', 'Der psychiater', 'Das haus', 'Die matratze', 'Der topf', 'Die handvoll', 'Das problem', 'Der abstieg', 'Das spritzen', 'Die struktur', 'Die wohnung', 'Die vermutung', 'Der daumen', 'Die kneipe', 'Die schachtel', 'Der clip', 'Der fall', 'Der zigeuner', 'Die öffnung', 'Die quelle', 'Das geheimnis', 'Der knopf', 'Die brise', 'Der rücken', 'Das kätzchen', 'Das schrotflinte', 'Das gesetz', 'Die wut', 'Die zigarre', 'Die zivilisation', 'Die rüstung', 'Der junge', 'Die bande', 'Das symptom', 'Der diakon', 'Die lodge', 'Die dame', 'Der tarif', 'Der löffel', 'Die verhaftung', 'Das signal', 'Die kartoffel', 'Der bildschirm', 'Der planet', 'Der instinkt', 'Der teich', 'Die schwalbe', 'Die leiche', 'Der status', 'Der dorn', 'Die generation', 'Die kamera', 'Der

In [66]:
upgradables = {'u':'ü', 'a':'ä', 'o':'ö', 'A':'Ä', 'O':'Ö', 'U':'Ü'}
def find_rule(noun):
    singular = noun[2].split(' ')[1]
    plural = noun[3].split(' ')[1]
    
    if len(singular) > len(plural):
        return '$' + plural
        
    counter = 0
    found_change = 0
    for i, letter in enumerate(singular):
        if letter in upgradables:
            counter += 1
        if plural[i] != letter:
            if letter not in upgradables or plural[i] != upgradables[letter] or found_change:
                formatting = singular[i:] + '->' + plural[i:]
                if counter == 0:
                    return '-' + formatting
                else:
                    return str(found_change) + formatting
            found_change = counter
    if counter == 0:
        return '-' + plural[len(singular):]
    return str(found_change) + plural[len(singular):]

    
def rule_extraction(data):
    plural_data = []
    for x in data:
        plural_data.append([*x, find_rule(x)])
    return plural_data
        

pluralized_data = rule_extraction(purged_plurality)

In [67]:
def find_suffixes(plurals):    
    parsed_suffixes = {}
    def add_suffix(suffix, element):
        if suffix not in parsed_suffixes:
            parsed_suffixes[suffix] = [element]
        else:
            parsed_suffixes[suffix].append(element)
    
    for x in pluralized_data:
        pattern = x[2].lower()
        add_suffix(pattern[-2:], x)
        add_suffix(pattern[-1:], x)
        if len(x) > 2:
            add_suffix(pattern[-3:], x)
    return parsed_suffixes
        
    
parsed_suffixes = find_suffixes(pluralized_data)

In [68]:
THRESHOLD = 2
better_suffixes = {}
for key in parsed_suffixes:
    if len(parsed_suffixes[key]) <= THRESHOLD:
        continue
    print(f'Key {key}:')
    for x in parsed_suffixes[key]:
        print(x)
    better_suffixes[key] = parsed_suffixes[key]
    print()

Key ei:
['437.', 'Egg', 'Das Ei', 'Die Eier', '-er']
['600.', 'File', 'Die Datei', 'Die Dateien', '0en']
['846.', 'Scream', 'Der Schrei', 'Die Schreie ', '-e']
['2959.', 'Bakery', 'Die Bäckerei', 'Die Bäckereien', '-en']

Key i:
['437.', 'Egg', 'Das Ei', 'Die Eier', '-er']
['2208.', 'Shark', 'Der Hai', 'Die Haie', '0e']
['2571.', 'Ski', 'Der Ski', 'Die Skis', '-s']
['751.', 'Cab', 'Das Taxi', 'Die Taxis ', '0s']
['600.', 'File', 'Die Datei', 'Die Dateien', '0en']
['1643.', 'Rabbi', 'Der Rabbi', '(plural form is Rabbiner)', '$form']
['2263.', 'Chili', 'Das Chili', 'Die Chilis', '-s']
['846.', 'Scream', 'Der Schrei', 'Die Schreie ', '-e']
['1738.', 'Gum', 'Der Kaugummi', 'Die Kaugummis', '0s']
['2959.', 'Bakery', 'Die Bäckerei', 'Die Bäckereien', '-en']

Key h:
['2048.', 'Bra', 'Der BH', 'Die BHs', '-s']
['762.', 'Cow', 'Die Kuh', 'Die Kühe ', '1e']
['1194.', 'Deer', 'Das Reh', 'Die Rehe', '-e']
['61.', 'Book', 'Das Buch', 'Die Bücher', '1er']
['251.', 'Hole', 'Das Loch', 'Die Löcher', '

In [87]:
def is_monosyllabic(word):
    syllables = 0
    vowel = False
    vowels = ['a', 'e', 'i', 'o', 'u', 'y', 'ä', 'ü', 'ö']
    for x in word:
        if x in vowels:
            if not vowel:
                syllables += 1
            vowel = True
        else:
            vowel = False
    return syllables == 1

def umlautifier(word):
    syllables = 0
    vowel = False
    vowels = ['a', 'o', 'u']
    for x in word:
        if x in vowels:
            return x
    return '-'

df = pd.DataFrame(pluralized_data, columns=('cardinal', 'english', 'g_singular', 'g_plural', 'rules'))
df['g_plural_nude'] = df['g_plural'].apply(lambda x: x[4:].strip().lower())
df['g_singular_nude'] = df['g_singular'].apply(lambda x: x[4:].strip().lower())
df['gender'] = df['g_singular'].apply(lambda x: x[:3])
df['p_ge'] = df['g_singular_nude'].apply(lambda x: True if x[4:6] == 'ge' else False)
df['monosyllabic'] = df['g_singular_nude'].apply(is_monosyllabic)
df['first_umlautifier'] = df['g_singular_nude'].apply(umlautifier)

for suffix in better_suffixes:
    df['s_' + suffix] = df['g_singular_nude'].apply(lambda x: True if x[-len(suffix):] == suffix else False)

/tmp/ipykernel_11771/1760875860.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['s_' + suffix] = df['g_singular_nude'].apply(lambda x: True if x[-len(suffix):] == suffix else False)
/tmp/ipykernel_11771/1760875860.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['s_' + suffix] = df['g_singular_nude'].apply(lambda x: True if x[-len(suffix):] == suffix else False)
/tmp/ipykernel_11771/1760875860.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many time

In [88]:
def execute_transform(word, mapping):
    mapped_word = word
    rule_type, transform = mapping
    if transform[0] == '$':
        return transform[1:]
    
    if transform[0] >= '1' and transform[0] <= '9':
        count = ord(transform[0]) - 48
        for (i, x) in enumerate(mapped_word):
            if x in upgradables:
                count -= 1
                if not count:
                    break
        if not count: #Error or reject as inconsistent?
            mapped_word = word[:i] + upgradables[x] + word[i+1:]

    
    new_transform = transform[1:]
    if rule_type == 'S':
        i=0
        while i < len(new_transform): #Trivial string matching - kmp unnecessary, length <= 3
            #print(new_transform[:len(new_transform)-i], mapped_word[len(mapped_word)-len(new_transform)+i:])
            if new_transform[:len(new_transform)-i] == mapped_word[len(mapped_word)-len(new_transform)+i:]:
                break
            i += 1
        return mapped_word[:len(mapped_word)-len(new_transform)+i] + new_transform
    
    if '->' in new_transform:
        left, right = new_transform.split('->')
        return mapped_word[:-len(left)] + right
    else:
        return mapped_word + new_transform

def filter_dataset_by_rule(rules, df):
    subset = pd.Series(True, index=df.index)
    for rule in rules:
        subset = subset & (df[rule] == rules[rule])
    return subset

def transform_dataset(transform, df, show_exceptions = False):
    inferred = df['g_singular_nude'].apply(lambda x: execute_transform(x, transform))
    ground = df['g_plural_nude']
    if show_exceptions:
        print(df[inferred != ground][['g_singular_nude', 'g_plural_nude']].to_string())
    return np.mean(inferred == ground)

def check_cover_dataset(rules, transform, df, show_exceptions = False):
    subset = filter_dataset_by_rule(rules, df)
    return transform_dataset(transform, df[subset], show_exceptions), len(df[subset])

rules = {'s_o': True}
check_cover_dataset(rules, ('T', '-s'), df)
#print(execute_transform('Eha', ('S', '-en')))

(np.float64(0.8333333333333334), 18)

In [89]:
NEW_THRESHOLD = 4

def find_transforms(df):
    usefuls = set(df['rules'].value_counts().nlargest(2).keys())
    to_add = set()
    for x in usefuls:
        if x[0] == '-':
            to_add.add('1' + x[1:])
    usefuls = {*usefuls, *to_add}
    usefuls = {('T', x) for x in usefuls}
    usefuls.add(('S', '-en'))
    return usefuls

def find_best_transform(df):
    transforms = find_transforms(df)
    current_best, supreme_transform = -1, '-'
    for transform in transforms:
        result = transform_dataset(transform, df)
        if result > current_best:
            current_best, supreme_transform = result, transform
    return current_best, supreme_transform    

def generate_transforms(df):
    all_results = []
    all_names = {*map(lambda x: 's_' + x, better_suffixes), 'p_ge', '0'}
    options = [(gender, syllable, first_umlautifier, name)
     for gender in ('Der', 'Die', 'Das', '0')
     for syllable in [True, False, None]
     for first_umlautifier in ['a', 'e', 'u', '-', None]
     for name in {*map(lambda x: 's_' + x, better_suffixes), 'p_ge', '0'}
    ]
    
    for option in options:
        rules = {}
        gender, syllable, first_umlautifier, name = option
        if gender != '0':
            rules['gender'] = gender
        if syllable != None:
            rules['monosyllabic'] = syllable
        if first_umlautifier != None:
            rules['first_umlautifier'] = first_umlautifier
        if name != '0':
            rules[name] = True
        subset = df[filter_dataset_by_rule(rules, df)]
        if len(subset) <= NEW_THRESHOLD:
            continue
        all_results.append([rules.copy(), len(subset), *find_best_transform(subset)])
    return all_results

all_results = generate_transforms(df)

In [93]:
def calculate_result_strength(result):
    strength = (result[1]**0.5)*(result[2]**4)
    for rule in result[0]:
        if rule == 'gender' or 'monosyllabic' or 'first_umlautifier':
            strength -= 0.5
        else:
            strength -= (len(rule)-2)*0.01
    return strength

new_results = []

for x in all_results:
    new_results.append((*x, calculate_result_strength(x)))

new_results = sorted(new_results, key=lambda x: -x[4])
for x in new_results:
    # if 'gender' not in x[0] or x[0]['gender'] != 'Die' or x[3] == ('S', '-en'):
    #     continue
    print(x)

#Good rules: 
# Die: -er -> T: -n
# Die: -> S: -en
# -o -> T: -s
# Der: -er -> T: -

({'gender': 'Die', 's_e': True}, 354, np.float64(0.9887005649717514), ('T', '0n'), np.float64(16.97880244462358))
({'gender': 'Die', 'monosyllabic': False, 's_e': True}, 350, np.float64(0.9914285714285714), ('T', '0n'), np.float64(16.57505984733688))
({'s_e': True}, 434, np.float64(0.9400921658986175), ('T', '0n'), np.float64(15.771462853207204))
({'monosyllabic': False, 's_e': True}, 427, np.float64(0.9461358313817331), ('T', '0n'), np.float64(15.558763288867183))
({'gender': 'Die', 'monosyllabic': False}, 673, np.float64(0.8900445765230312), ('S', '-en'), np.float64(15.280006155971492))
({'gender': 'Die'}, 735, np.float64(0.8585034013605443), ('S', '-en'), np.float64(14.226913982434676))
({'gender': 'Die', 'first_umlautifier': '-', 's_e': True}, 138, np.float64(0.9927536231884058), ('T', '1n'), np.float64(9.910520780124962))
({'gender': 'Die', 's_g': True}, 119, np.float64(1.0), ('S', '-en'), np.float64(9.908712114635714))
({'gender': 'Die', 's_ng': True}, 119, np.float64(1.0), ('S',

In [94]:
def check_rule_cover(covering, covered):
    suffix = [x for x in covered if x[0] == 's']
    suffix = suffix[0][2:] if suffix else ''
        
    for x in covering:
        if x[0] != 's':
            if x not in covered or covered[x] != covering[x]:
                return False
        elif not suffix.endswith(x[2:]):
            return False
    return True

def extract_suffix(rule):
    for x in rule:
        if x.startswith('s_'):
            return x[2:]
            
def independent(rule_1, rule_2):
    if 'gender' in rule_2 and 'gender' in rule_1 and rule_2['gender'] != rule_1['gender']:
        return True
    
    s1, s2 = extract_suffix(rule_1), extract_suffix(rule_2)
    if s1 and s2 and not s1.endswith(s2) and not s2.endswith(s1):
        return True
    
    return False

In [95]:
import time

def reevaluate_rules(all_rules, new_rule, df):
    new_rules = []
    for x in all_rules:
        if x[0] == {'s_pt': True}:
            print('A', len(df[filter_dataset_by_rule({'s_pt': True}, df)]))
        if not x[0]:
            continue
        if x[0] == {'s_pt': True}:
            print('B', len(df[filter_dataset_by_rule({'s_pt': True}, df)]))
        if check_rule_cover(new_rule[0], x[0]):
            continue
        if x[0] == {'s_pt': True}:
            print('C', len(df[filter_dataset_by_rule({'s_pt': True}, df)]))
        if independent(new_rule[0], x[0]):
            new_element = [*x]
        else:
            subset = df[filter_dataset_by_rule(x[0], df)]
            if len(subset) <= NEW_THRESHOLD:
                continue
            new_element = [x[0].copy(), len(subset), *find_best_transform(subset)]
        new_rules.append([*new_element, calculate_result_strength(new_element)])
    return new_rules

def framework(df, results):
    i=0
    all_rules = []
    while len(results):
        good_rule = results[0]
        print(i, 'rule: ' + str(good_rule[0]), 'cover: ' + str(good_rule[1]), 'accuracy: ' + str(good_rule[2]), 'rule: ' + str(good_rule[3]), 'cost_function: ' + str(good_rule[4]), 'dataset_size: ' + str(len(df)), sep='\n')
        print()
        all_rules.append(good_rule)
        df = df[np.invert(filter_dataset_by_rule(good_rule[0], df))]
        results = sorted(reevaluate_rules(results, good_rule, df), key=lambda x:-x[4])
        i += 1
    return all_rules

t1 = time.time()
all_rules = framework(df.copy(), new_results)
print(time.time()-t1)

0
rule: {'gender': 'Die', 's_e': True}
cover: 354
accuracy: 0.9887005649717514
rule: ('T', '0n')
cost_function: 16.97880244462358
dataset_size: 1910

1
rule: {'gender': 'Die', 's_g': True}
cover: 119
accuracy: 1.0
rule: ('S', '-en')
cost_function: 9.908712114635714
dataset_size: 1556

2
rule: {'gender': 'Der', 'monosyllabic': False, 's_er': True}
cover: 162
accuracy: 0.9444444444444444
rule: ('T', '-')
cost_function: 8.62658872967792
dataset_size: 1437

3
rule: {'gender': 'Die', 's_on': True}
cover: 50
accuracy: 1.0
rule: ('S', '-en')
cost_function: 6.0710678118654755
dataset_size: 1275

4
rule: {'s_eit': True}
cover: 26
accuracy: 1.0
rule: ('T', '-en')
cost_function: 4.5990195135927845
dataset_size: 1225

5
rule: {'s_en': True}
cover: 102
accuracy: 0.8333333333333334
rule: ('T', '-')
cost_function: 4.370517427836651
dataset_size: 1199

6
rule: {'gender': 'Der', 'monosyllabic': False, 'first_umlautifier': '-', 's_el': True}
cover: 31
accuracy: 1.0
rule: ('T', '-')
cost_function: 3.5677

In [16]:
print(all_rules)

NameError: name 'all_rules' is not defined

In [101]:
#df[filter_dataset_by_rule({'gender': 'Die', 's_e': True}, df)]
filtered_df = df[df['s_er'] == False]
filtered_df = filtered_df[filtered_df['s_e'] == False]
filtered_df = filtered_df[filtered_df['s_el'] == False]
check_cover_dataset({'first_umlautifier': '-', 'gender':'Der'}, ('S', '-e'), filtered_df, True){'gender': 'Der', 'monosyllabic': False, 'first_umlautifier': '-', 's_el': True}

     g_singular_nude g_plural_nude
1                 bh           bhs
28               bär         bären
39               typ         typen
45               hit          hits
48               elf         elfen
53               kip          kips
56               ski          skis
61               til          tils
105             herr        herren
166             test         tests
167             chef         chefs
173             tipp         tipps
188             held        helden
192             chip         chips
193             fick         ficks
199             nerv        nerven
236             jeep         jeeps
243             link         links
249             fels        felsen
281             clip         clips
298             slip         slips
354            regen         regen
373            fleck       flecken
383            geist       geister
443            prinz       prinzen
471            trick        tricks
556            klick        klicks
598            segen

(np.float64(0.6136363636363636), 132)

In [54]:
df[['g_singular_nude', 'first_umlautifier']]

,g_singular_nude,first_umlautifier
0,ei,None
1,bh,None
2,tag,a
3,weg,None
4,tür,None
5,ort,o
6,arm,a
7,fuß,u
8,job,o
9,tod,o


In [50]:
[(gender, syllable, first_umlautifier, name) 
 for gender in ('Der', 'Die', 'Das', '0')
 for syllable in [True, False, None]
 for first_umlautifier in ['a', 'e', 'u']
 for name in {*map(lambda x: 's_' + x, better_suffixes), 'p_ge', '0'}
]

[('Der', True, 'a', 's_lf'),
 ('Der', True, 'a', 's_ium'),
 ('Der', True, 'a', 's_uch'),
 ('Der', True, 'a', 's_a'),
 ('Der', True, 'a', 's_est'),
 ('Der', True, 'a', 's_orm'),
 ('Der', True, 'a', 's_rt'),
 ('Der', True, 'a', 's_uer'),
 ('Der', True, 'a', 's_ube'),
 ('Der', True, 'a', 's_amm'),
 ('Der', True, 'a', 's_nz'),
 ('Der', True, 'a', 's_fen'),
 ('Der', True, 'a', 's_rk'),
 ('Der', True, 'a', 's_pf'),
 ('Der', True, 'a', 's_age'),
 ('Der', True, 'a', 's_sen'),
 ('Der', True, 'a', 's_itz'),
 ('Der', True, 'a', 's_ife'),
 ('Der', True, 'a', 's_he'),
 ('Der', True, 'a', 's_tur'),
 ('Der', True, 'a', 's_g'),
 ('Der', True, 'a', 's_ume'),
 ('Der', True, 'a', 's_her'),
 ('Der', True, 'a', 's_kel'),
 ('Der', True, 'a', 's_ht'),
 ('Der', True, 'a', 's_gel'),
 ('Der', True, 'a', 's_eug'),
 ('Der', True, 'a', 's_ing'),
 ('Der', True, 'a', 's_nn'),
 ('Der', True, 'a', 's_ier'),
 ('Der', True, 'a', 's_ffe'),
 ('Der', True, 'a', 's_cke'),
 ('Der', True, 'a', 's_o'),
 ('Der', True, 'a', 's_e

In [169]:
print(df.head(50))

   cardinal     english g_singular        g_plural     rules g_plural_nude  \
0      437.         Egg     Das Ei        Die Eier       -er          eier   
1     2048.         Bra     Der BH         Die BHs        -s           bhs   
2        4.         Day    Der Tag        Die Tage        0e          tage   
3        5.         Way    Der Weg        Die Wege        -e          wege   
4       11.        Door    Die Tür       Die Türen       -en         türen   
5       23.       Place    Der Ort        Die Orte        0e          orte   
6       27.         Arm    Der Arm        Die Arme        0e          arme   
7       32.        Foot    Der Fuß        Die Füße        1e          füße   
8       89.         Job    Der Job        Die Jobs        0s          jobs   
9       97.       Death    Der Tod        Die Tode        0e          tode   
10     133.         Ear    Das Ohr       Die Ohren       0en         ohren   
11     153.         Bar    Die Bar        Die Bars        0s    

In [107]:
def framework(df, results): #Brute for testing
    for i in range(1):
        good_rule = results[0]
        print(good_rule)
        df = df[np.invert(filter_dataset_by_rule(good_rule[0], df))]
        results = sorted(reevaluate_rules(results, good_rule, df), key=lambda x:-x[4])
        for x in results:
            print(x)

In [17]:
#check_cover_dataset({'gender': 'Der', 'monosyllabic':True, 's_er': True}, ('T', '-'), df, True)
#check_cover_dataset({'s_e': True}, ('S', '-en'), df, True)
#check_cover_dataset({'gender':'Die'}, ('S', '-en'), df, True)
#check_cover_dataset({'gender': 'Die', 's_er': True}, ('T', '-n'), df, True)
check_cover_dataset({'s_lt': True}, ('T', '-n'), df, True)

     g_singular_nude g_plural_nude
74              welt        welten
158             halt         halte
189             zelt         zelte
771           anwalt       anwälte
864           inhalt       inhalte
1013          gehalt      gehälter
1411        vielfalt    vielfalten


(np.float64(0.0), 7)

In [14]:
print(set(df['rules']))

{'$Fälle', '-phin->fine', '-n', '-', '1er', '0ta', '0schläge', '0os->en', '-ien', '0en->ungen', '0se', '0nen', '-en', '0e', '0f->be', '-Pelz->Felle', '-re->nräume', '-en->ungen', '-innen', '0sagungen', '0arten', '0el->le', '$Kicher', '1e', '0n', '0r->n', '2e', '0en->ina', '$Campi', '-Butler->Diener', '-s', '$Maxima', '0es', '0s', '-nisse', '-er', '0en', '0a->en', '1al->le', '0o->en', '-nen', '0is->en', '0um->en', '0', '$Flüster', '0er', '-ex->izes', '$Vakua', '-se', '-e', '1', '0us->en', '0ien', '$Stati', '0ps->es', '$Reiche', '$Foki', '3e', '$Stöhne', '-Post->Mails', '$form'}


In [15]:
print(pluralized_data)

[['437.', 'Egg', 'Das Ei', 'Die Eier', '-er'], ['2048.', 'Bra', 'Der BH', 'Die BHs', '-s'], ['4.', 'Day', 'Der Tag', 'Die Tage', '0e'], ['5.', 'Way', 'Der Weg', 'Die Wege', '-e'], ['11.', 'Door', 'Die Tür', 'Die Türen', '-en'], ['23.', 'Place', 'Der Ort', 'Die Orte', '0e'], ['27.', 'Arm', 'Der Arm', 'Die Arme', '0e'], ['32.', 'Foot', 'Der Fuß', 'Die Füße', '1e'], ['89.', 'Job', 'Der Job', 'Die Jobs', '0s'], ['97.', 'Death', 'Der Tod', 'Die Tode', '0e'], ['133.', 'Ear', 'Das Ohr', 'Die Ohren', '0en'], ['153.', 'Bar', 'Die Bar', 'Die Bars', '0s'], ['236.', 'Yard', 'Der Hof', 'Die Höfe', '1e'], ['244.', 'Ice', 'Das Eis', 'Die Eis', '-'], ['258.', 'Bit', 'Das Bit', 'Die Bits', '-s'], ['273.', 'Hat', 'Der Hut', 'Die Hüte', '1e'], ['299.', 'Train', 'Der Zug', 'Die Züge', '1e'], ['306.', 'Bus', 'Der Bus', 'Die Busse', '0se'], ['350.', 'Gate', 'Das Tor', 'Die Tore', '0e'], ['359.', 'Wheel', 'Das Rad', 'Die Räder', '1er'], ['364.', 'Tea', 'Der Tee', 'Die Tees', '-s'], ['368.', 'Tone', 'Der Ton'

In [16]:
curios = [x for x in pluralized_data if x[2].endswith('um')]
gestuff = [x for x in pluralized_data if x[4].startswith('$')]

In [17]:
for x in curios:
    print(x)
print(len(curios))

['66.', 'Tree', 'Der Baum', 'Die Bäume', '1e']
['150.', 'Space', 'Der Raum', 'Die Räume', '1e']
['2349.', 'Hem', 'Der Saum', 'Die Säume', '1e']
['154.', 'Dream', 'Der Traum', 'Die Träume', '1e']
['475.', 'Date', 'Das Datum', 'Die Daten', '0um->en']
['2153.', 'Album', 'Das Album', 'Die Alben', '0um->en']
['1075.', 'Museum', 'Das Museum', 'Die Museen', '0um->en']
['2341.', 'Vacuum', 'Das Vakuum', 'Die Vakua', '$Vakua']
['2537.', 'Foam', 'Der Schaum', 'Die Schäume', '1e']
['2907.', 'Podium', 'Das Podium', 'Die Podien', '0um->en']
['205.', 'Center', 'Das Zentrum', 'Die Zentren', '0um->en']
['339.', 'Maximum', 'Das Maximum', 'Die Maxima', '$Maxima']
['528.', 'Study', 'Das Studium', 'Die Studien', '0um->en']
['1811.', 'Empire', 'Das Imperium', 'Die Reiche', '$Reiche']
['2444.', 'Anniversary', 'Das Jubiläum', 'Die Jubiläen', '0um->en']
['2678.', 'Refuge', 'Das Refugium', 'Die Refugien', '0um->en']
['2930.', 'Belongings', 'Das Eigentum', 'Die Eigentum', '0']
['809.', 'Universe', 'Das Universum

In [18]:
print(len(pluralized_data))
print(len({x[4] for x in pluralized_data}))

1912
61


In [19]:
_x = {x[4] for x in pluralized_data if x[4][0]!='$'}
print(_x)

{'-phin->fine', '-n', '-', '1er', '0ta', '0schläge', '0os->en', '-ien', '0en->ungen', '0se', '0nen', '-en', '0e', '0f->be', '-Pelz->Felle', '-re->nräume', '-en->ungen', '-innen', '0sagungen', '0arten', '0el->le', '1e', '0n', '0r->n', '2e', '0en->ina', '-Butler->Diener', '-s', '0es', '0s', '-nisse', '-er', '0en', '0a->en', '1al->le', '0o->en', '-nen', '0is->en', '0um->en', '0', '-ex->izes', '-se', '-e', '1', '0us->en', '0ien', '0ps->es', '3e', '-Post->Mails', '0er'}
